In [1]:
import torch
print(torch.__version__)

print(torch.cuda.is_available())
print(torch.cuda.device_count())



2.14.1+cu130
True
1


In [2]:
import torch.nn as nn
import matplotlib.pyplot as plt

def make_generator_network(input_size = 20, num_hidden_layers =1, num_hidden_units = 100, num_output_units= 784):
  model =nn.Sequential()
  for i in range(num_hidden_layers):
    model.add_module(f"linear{i}", nn.Linear(input_size, num_hidden_units))
    model.add_module(f"acttivation_func{i}", nn.LeakyReLU())
    input_size = num_hidden_units
  model.add_module("output_layer", nn.Linear(input_size, num_output_units))

  model.add_module("tanh", nn.Tanh())

  return model





In [3]:
def make_discriminator_network(input_size = 784, num_hidden_layers = 1, num_hidden_units = 100, num_output_units =1):
  model = nn.Sequential()
  for i in range(num_hidden_layers):
    model.add_module(f"linear{i}", nn.Linear(input_size, num_hidden_units))
    model.add_module(f"acttivation_func{i}", nn.LeakyReLU())
    model.add_module("dropout", nn.Dropout(p=0.5))
    input_size = num_hidden_units
  model.add_module("output_layer", nn.Linear(input_size, num_output_units))
  model.add_module('sigmoid', nn.Sigmoid())
  return model

In [17]:
import numpy as np
image_size = (28,28)
z_size = 20
gen_hidden_layer = 1
gen_hidden_size = 100
discr_hidden_layer =1
discr_hidden_size = 100

torch.manual_seed(2)

gen_model = make_generator_network(input_size = z_size,num_hidden_layers = gen_hidden_layer,num_hidden_units= gen_hidden_size, num_output_units= np.prod(image_size))
discr_model = make_discriminator_network(input_size = np.prod(image_size), num_hidden_units = discr_hidden_size, num_hidden_layers= discr_hidden_layer)

print(gen_model)
print(discr_model)


Sequential(
  (linear0): Linear(in_features=20, out_features=100, bias=True)
  (acttivation_func0): LeakyReLU(negative_slope=0.01)
  (output_layer): Linear(in_features=100, out_features=784, bias=True)
  (tanh): Tanh()
)
Sequential(
  (linear0): Linear(in_features=784, out_features=100, bias=True)
  (acttivation_func0): LeakyReLU(negative_slope=0.01)
  (dropout): Dropout(p=0.5, inplace=False)
  (output_layer): Linear(in_features=100, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


# Defining the training dataset
    


In [18]:
import torchvision
from torchvision import transforms

image_path ="/"
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize(mean = (0.5), std = (0.5)),
     ]
)
mnist_dataset = torchvision.datasets.MNIST(
    root = image_path,
    train = True,
    download = True,
    transform = transform
)
example, label = next(iter(mnist_dataset))

print(f'Min: {example.min()} Max: {example.max()}')
print(example.shape)

Min: -1.0 Max: 1.0
torch.Size([1, 28, 28])


In [19]:
# noise generation
def create_noise(batch_size, z_size, mode_x):
  if mode_x == 'normal':
    input_z = torch.randn(batch_size, z_size)*2 -1
  elif mode_x == 'uniform':
    input_z = torch.rand(batch_size, z_size)
  return input_z
  #return torch.randn(batch_size, z_size)

In [20]:
from torch.utils.data import DataLoader
batch_size = 32
dataloader = DataLoader(mnist_dataset, batch_size =32, shuffle = True)
input_real, label = next(iter(dataloader))
input_real = input_real.view(batch_size, -1)

torch.manual_seed(1)
mode_z = 'uniform'

input_z = create_noise(batch_size, z_size, mode_z)
#x_size dfn ealier
#print(input_z.shape)

In [21]:


g_output = gen_model(input_z) # outputs generated fake images
d_proba_real = discr_model(input_real) # processes real images
d_proba_fake = discr_model(g_output) # processes fake images

print('input-z -- shape:', input_z.shape)
print('input-real -- shape:', input_real.shape)
print('Output of G -- shape:', g_output.shape)
print('Disc. (real) -- shape:', d_proba_real.shape)
print('Disc. (fake) -- shape:', d_proba_fake.shape)

input-z -- shape: torch.Size([32, 20])
input-real -- shape: torch.Size([32, 784])
Output of G -- shape: torch.Size([32, 784])
Disc. (real) -- shape: torch.Size([32, 1])
Disc. (fake) -- shape: torch.Size([32, 1])
